# Analisis Regresi Linear Berganda untuk Efisiensi Bahan Bakar (Auto MPG)
Notebook ini berfungsi sebagai mesin pemrosesan data, pemodelan statistik, dan pembuatan grafik visualisasi untuk proyek analisis efisiensi bahan bakar.

## 1. Impor Pustaka

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

## 2. Membaca dan Membersihkan Data

In [ ]:
# Membaca dataset mentah
df = pd.read_csv('data/auto_mpg_raw.csv')

# Merapikan nama kolom
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

# Memastikan kolom target dan prediktor bertipe data numerik
numeric_columns = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Menghapus baris yang mengandung nilai kosong (NaN)
df = df[numeric_columns].dropna()

# Menyimpan dataset yang bersih ke folder data
df.to_csv('data/auto_mpg_clean.csv', index=False)
print(f"Data bersih berhasil disimpan ke data/auto_mpg_clean.csv. Jumlah baris: {len(df)}")
df.head()

## 3. Perhitungan Korelasi

In [ ]:
variables = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
correlation_matrix = df[variables].corr()
print("Matriks Korelasi:")
print(correlation_matrix)

## 4. Forward Stepwise Selection (dengan Uji F)

In [ ]:
variable_names = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
X = df[variable_names].values
y = df['mpg'].values

remaining = list(range(len(variable_names)))
selected = []

# Forward selection berbasis signifikansi Uji F
while remaining:
    best_f = -1
    best_idx = None
    for idx in remaining:
        X_temp = X[:, selected + [idx]]
        X_temp = np.column_stack([np.ones(len(X_temp)), X_temp])
        beta_temp = np.linalg.inv(X_temp.T @ X_temp) @ (X_temp.T @ y)
        y_pred_temp = X_temp @ beta_temp
        n = len(y)
        k = len(selected) + 1
        sse = np.sum((y - y_pred_temp)**2)
        sst = np.sum((y - np.mean(y))**2)
        ssr = sst - sse
        f = (ssr / k) / (sse / (n - k - 1))
        print(f"{variable_names[idx]} -> F = {f:.4f}")
        if f > best_f:
            best_f = f
            best_idx = idx
    
    if best_f > 4:
        selected.append(best_idx)
        remaining.remove(best_idx)
        print(f"Menambahkan: {variable_names[best_idx]}\n")
    else:
        break

print('Variabel terpilih:')
for i in selected:
    print('-', variable_names[i])

X_final = X[:, selected]
X_final = np.column_stack([np.ones(len(X_final)), X_final])

beta = np.linalg.inv(X_final.T @ X_final) @ (X_final.T @ y)
ypred = X_final @ beta

## 5. Interpretasi Koefisien Regresi

In [ ]:
print(f'Intercept = {beta[0]:.4f}\n')

for i, idx in enumerate(selected):
    name = variable_names[idx]
    print(f'{name}: {beta[i+1]:.4f}')

## 6. Uji Kecocokan Model (R² & Uji F)

In [ ]:
n = len(y)
k = len(selected)

sst = np.sum((y - np.mean(y))**2)
sse = np.sum((y - ypred)**2)
ssr = sst - sse

r2 = ssr / sst
adj_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))
f_stat = (ssr / k) / (sse / (n - k - 1))

print(f'R² = {r2:.4f} ({r2*100:.2f}%)')
print(f'Adjusted R² = {adj_r2:.4f}')
print(f'F-statistic = {f_stat:.4f}')

## 7. Analisis Lanjutan dan Pembuatan Grafik (300 DPI)

In [ ]:
corr_target = correlation_matrix['mpg']
print('Korelasi terhadap mpg:')
print(corr_target.sort_values(ascending=False))

print('\nPersamaan Regresi:')
print(f'mpg = {beta[0]:.4f}', end=' ')
for i, idx in enumerate(selected):
    print(f'+ ({beta[i+1]:.4f})*{variable_names[idx]}', end=' ')
print()

X_std = (X - X.mean(axis=0)) / X.std(axis=0)
y_std = (y - y.mean()) / y.std()
Xs = np.column_stack([np.ones(len(X_std)), X_std])
beta_std = np.linalg.inv(Xs.T @ Xs) @ (Xs.T @ y_std)

print('\nKoefisien Standar:')
for i, name in enumerate(variable_names):
    print(name, ':', round(beta_std[i+1], 4))

# Pembuatan Folder Gambar jika belum ada
os.makedirs('images', exist_ok=True)

# 1. Heatmap Korelasi
plt.figure(figsize=(8, 6), dpi=300)
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Heatmap Korelasi antar Fitur', fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('images/correlation_heatmap.png', dpi=300)
plt.close()

# 2. Hubungan Bobot Kendaraan vs. Efisiensi (MPG)
plt.figure(figsize=(8, 6), dpi=300)
sns.regplot(data=df, x='weight', y='mpg', 
            scatter_kws={'alpha':0.6, 'color':'royalblue', 'edgecolor':'w'},
            line_kws={'color':'crimson', 'linewidth':2, 'label':'Garis Tren Regresi'})
plt.title('Hubungan Bobot Kendaraan vs. Efisiensi (MPG)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Weight (Bobot Kendaraan - lbs)')
plt.ylabel('MPG (Miles Per Gallon)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig('images/mpg_vs_weight.png', dpi=300)
plt.close()

# 3. Nilai Aktual vs. Prediksi MPG
plt.figure(figsize=(8, 6), dpi=300)
plt.scatter(y, ypred, color='forestgreen', alpha=0.6, edgecolors='w', label='Prediksi vs Aktual')
min_val = min(min(y), min(ypred))
max_val = max(max(y), max(ypred))
plt.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=2, label='Ideal (y = x)')
plt.title('Nilai Aktual vs. Prediksi MPG', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Actual MPG (Nilai Aktual)')
plt.ylabel('Predicted MPG (Nilai Prediksi)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/actual_vs_predicted.png', dpi=300)
plt.close()

# 4. Grafik Residual (Residuals Plot)
residuals = y - ypred
plt.figure(figsize=(8, 6), dpi=300)
plt.scatter(ypred, residuals, color='darkorange', alpha=0.6, edgecolors='w')
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.title('Grafik Residual (Residuals Plot)', fontsize=12, fontweight='bold', pad=15)
plt.xlabel('Predicted MPG (Nilai Prediksi)')
plt.ylabel('Residuals (Sisaan)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('images/residuals_plot.png', dpi=300)
plt.close()

print("Semua grafik berhasil disimpan ke folder 'images/' dengan resolusi 300 DPI.")